# AIPV1EMI0001 — rebuild and deploy on an EdgeAI Station

Step-by-step guide to rebuild **AIPV1EMI0001** (YOLOv7-tiny `car` / `person` / `truck` detector)
from this repository, register it on an EdgeAI Station, verify its callback output, and install
its Node-RED dashboard. Run the cells top to bottom; each section ends with a check.

**Where to run this notebook:** in the station's Jupyter environment (`ms8_jupyter`, the
App-Builder container). It has `eais-tools`, the CUDA toolkit, DeepStream headers and `trtexec`,
and it shares `/data/aip` (AI processors) and `/data/projects` (Node-RED projects) with the
station services, so every step runs in place.

| Step | What happens |
|---|---|
| 0 | Check the environment and the AIP folder location |
| 1 | Download the model from Hugging Face (ONNX, optionally the prebuilt engine) |
| 2 | Build the custom parser `.so` and the TensorRT engine (`bin/build.sh`) |
| 3 | Validate and package with `eais-tools` |
| 4 | Register the AIP with the video-streaming service (ms3) |
| 5 | Run a test pipeline and check the callback output contract |
| 6 | Install the Node-RED flow and the UIBuilder dashboard |
| 7 | Clean up |

## 0. Prerequisites

The station only discovers AI processors under `/data/aip/<unique_id>`, so this repository must be
cloned **as `/data/aip/AIPV1EMI0001`**. From a Jupyter terminal:

```bash
cd /data/aip
git clone https://github.com/edge-ai/AIPV1EMI0001.git
```

Then open `/data/aip/AIPV1EMI0001/localdev.ipynb` (this notebook) from that folder.

If `/data/aip/AIPV1EMI0001` already exists (an earlier install), move it aside first, for example
`mv AIPV1EMI0001 AIPV1EMI0001.backup.$(date +%s)`.

The next cell checks the location and the tools. The tested environment is JetPack 6.0 /
L4T R36.3 / TensorRT 8.6.2 / DeepStream 7.0 with `eais-tools` 0.9.2.

In [ ]:
import os, re, shutil, subprocess
from pathlib import Path

def run(args, *, cwd=None, env=None, expected=(0,)):
    result = subprocess.run(args, cwd=cwd, env=env)
    if result.returncode not in expected:
        raise subprocess.CalledProcessError(result.returncode, args)
    return result

AIP_ID = "AIPV1EMI0001"
AIP_ROOT = Path.cwd()
TRTEXEC = Path(os.environ.get("TRTEXEC_BIN", "/usr/src/tensorrt/bin/trtexec"))

assert AIP_ROOT == Path("/data/aip") / AIP_ID, (
    f"Run this notebook from /data/aip/{AIP_ID} (current directory: {AIP_ROOT})")
assert (AIP_ROOT / "manifest.json").is_file(), "manifest.json not found: is this the repository root?"

for tool in ("eais-tools", "make", "git", "npm", "timeout"):
    assert shutil.which(tool), f"{tool} not found in PATH"
assert Path("/usr/local/cuda/bin/nvcc").is_file(), "CUDA toolkit (nvcc) not found"
assert Path("/opt/nvidia/deepstream/deepstream/sources/includes").is_dir(), "DeepStream headers not found"
assert TRTEXEC.is_file(), f"trtexec not found at {TRTEXEC} (set TRTEXEC_BIN)"

trt_banner = subprocess.run([str(TRTEXEC), "--help"], capture_output=True, text=True).stdout
TRT_VERSION = (re.search(r"TensorRT v(\d+)", trt_banner) or [None, "unknown"])[1]
L4T = Path("/etc/nv_tegra_release").read_text().split(",")[:2] if Path("/etc/nv_tegra_release").exists() else ["unknown"]
free_gb = shutil.disk_usage(AIP_ROOT).free / 1e9

print(f"AIP folder    : {AIP_ROOT}")
print(f"TensorRT      : v{TRT_VERSION}")
print(f"L4T           : {' '.join(s.strip() for s in L4T)}")
print(f"Free disk     : {free_gb:.1f} GB")
assert free_gb > 1, "Less than 1 GB free on /data/aip"
print("OK: environment ready")

## 1. Download the model

The weights are **not stored in this repository**. They are published on Hugging Face at
[`Edgematrix-JP/AIPV1EMI0001`](https://huggingface.co/Edgematrix-JP/AIPV1EMI0001) and verified
here with SHA-256. The weights are licensed under GPL-3.0, matching the YOLOv7 training code.

Choose how to get the TensorRT engine:

- `ENGINE_SOURCE = "build"` (default) downloads the **ONNX** and builds the engine on this station
  in step 2 (allow 30–60 minutes on a busy Orin NX). The verified target is JetPack 6.0 /
  L4T R36.3 / TensorRT 8.6.2 / DeepStream 7.0; other SDK versions need separate validation.
- `ENGINE_SOURCE = "prebuilt"` downloads the published FP16 engine (dynamic batch 1–8). TensorRT
  engines only load on the TensorRT version they were built with, so this option is limited to
  **JetPack 6 / L4T R36.3 / TensorRT 8.6.2**.

In [ ]:
import hashlib, urllib.request

ENGINE_SOURCE = "build"  # "build" or "prebuilt"

HF_BASE = "https://huggingface.co/Edgematrix-JP/AIPV1EMI0001/resolve/main"
MODEL_DIR = AIP_ROOT / "models" / "AIPV1EMI0001_Yolo_Detector"
ENGINE_PATH = MODEL_DIR / "1" / "yolov7-fp16-1x8x8.engine"
PREBUILT_TRT_VERSION = "8602"

# Hugging Face file -> (destination, sha256)
ONNX = {"yolov7tiny-parking-rev3-640.onnx": (
    MODEL_DIR / "yolov7tiny-parking-rev3-640.onnx",
    "5796e9e0836751e3b7a9623cc6db976ac7fd087fcfb2742d20d06dcf0a0b3560")}
PREBUILT = {"yolov7-fp16-1x8x8-jp6-l4t36.3-arm64.engine": (
    ENGINE_PATH,
    "670150b495a0f120d178fbbb1e3f29758ccbee710e4477bf6847f46517a873b8")}

assert ENGINE_SOURCE in ("build", "prebuilt"), ENGINE_SOURCE
if ENGINE_SOURCE == "prebuilt":
    assert TRT_VERSION == PREBUILT_TRT_VERSION, (
        f"The prebuilt engine needs TensorRT v{PREBUILT_TRT_VERSION}, this station has v{TRT_VERSION}. "
        "Use ENGINE_SOURCE = 'build'.")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

for name, (dest, expected) in (ONNX if ENGINE_SOURCE == "build" else PREBUILT).items():
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and sha256(dest) == expected:
        print(f"OK (cached): {dest.relative_to(AIP_ROOT)}")
        continue
    print(f"Downloading {name} ...")
    urllib.request.urlretrieve(f"{HF_BASE}/{name}", dest)
    actual = sha256(dest)
    if actual != expected:
        dest.unlink()
        raise AssertionError(f"checksum mismatch for {name}: {actual}")
    print(f"OK: {dest.relative_to(AIP_ROOT)}")

## 2. Build the runtime artifacts

`bin/build.sh` produces the two binary files the AIP needs at runtime:

| Artifact | How |
|---|---|
| `models/.../libnvds_infercustomparser.so` | `make` in `src/nvdsinfer_customparser/` (DeepStream bounding-box parser for the `EfficientNMS_TRT` output) |
| `models/.../1/yolov7-fp16-1x8x8.engine` | `trtexec` FP16 build from the ONNX (dynamic batch 1–8, 640×640). Skipped with the prebuilt engine. |

**Memory note (`--workspace`):** trtexec's workspace is the GPU scratch memory used *during the build*.
Higher values let TensorRT consider more (faster) tactics. On a 16 GB Jetson Orin running the full
EdgeAI stack, 4096 MiB gets the build killed for lack of memory, so the script defaults to
**1024 MiB**. Override with `WORKSPACE_MIB=4096` on a quiet machine.

In [ ]:
skip = "1" if ENGINE_SOURCE == "prebuilt" else "0"
run(["bash", "./build.sh"], cwd=AIP_ROOT / "bin",
    env={**os.environ, "SKIP_ENGINE": skip})

In [ ]:
for path in (MODEL_DIR / "libnvds_infercustomparser.so", ENGINE_PATH):
    assert path.is_file() and path.stat().st_size > 0, f"missing build output: {path}"
    print(f"OK: {path.relative_to(AIP_ROOT)} ({path.stat().st_size / 1e6:.1f} MB)")
run([str(TRTEXEC), f"--loadEngine={ENGINE_PATH}", "--shapes=images:1x3x640x640",
     "--duration=1", "--warmUp=200", "--iterations=1"])
print("OK: engine deserializes and runs inference on this station")

## 3. Validate and package with eais-tools

Packaging is done with **`eais-tools`**, never by copying files by hand. `aip build` assembles
`dist/` with the runtime files only (engine, `.so`, configs, callback, manifest, docs, logo) plus
the Node-RED flow and dashboard listed in `.include`. Source files (`.onnx`, `.cpp`, `Makefile`,
this notebook) are left out.

The check after the build asserts the files the DeepStream/Triton configs reference. Without them
`eais-tools aip check` can still pass, but the pipeline fails to start.

In [ ]:
# Validate the AIP root folder (manifest, callback, model repository layout)
run(["eais-tools", "aip", "check", "--uuid", AIP_ID])

In [ ]:
# Assemble dist/ (runtime files only)
DIST = AIP_ROOT / "dist"
if DIST.exists():
    shutil.rmtree(DIST)
run(["eais-tools", "aip", "build", "--uuid", AIP_ID])

In [ ]:
DIST = AIP_ROOT / "dist"
required = [
    "manifest.json",
    "callback.py",
    "LICENSE",
    "LICENSE.weights",
    "README.md",
    "CHANGELOG.md",
    "resources/logo.png",
    "doc/USER_MANUAL.md",
    "configs/config_infer_primary_detector_yolov7.txt",
    "configs/config_tracker_NvDCF_perf.yml",
    "models/AIPV1EMI0001_Yolo_Detector/config.pbtxt",
    "models/AIPV1EMI0001_Yolo_Detector/labels-rev3.txt",
    "models/AIPV1EMI0001_Yolo_Detector/libnvds_infercustomparser.so",
    "models/AIPV1EMI0001_Yolo_Detector/1/yolov7-fp16-1x8x8.engine",
    "logic/flows.json",
    "logic/package.json",
    "logic/package-lock.json",
    "dashboard/package.json",
    "dashboard/package-lock.json",
    "dashboard/aipv1emi0001_dash/package.json",
    "dashboard/aipv1emi0001_dash/src/index.html",
    "dashboard/aipv1emi0001_dash/src/index.js",
    "dashboard/aipv1emi0001_dash/src/index.css",
]
missing = [p for p in required if not (DIST / p).is_file() or not (DIST / p).stat().st_size]
assert not missing, f"missing from dist/: {missing}"

forbidden = [p for p in DIST.rglob("*")
             if p.suffix in {".onnx", ".pt", ".cpp", ".ipynb"} or p.name in {"Makefile", "flows_cred.json"}
             or "node_modules" in p.parts]
assert not forbidden, f"files that must not ship: {forbidden}"

for p in sorted(DIST.rglob("*")):
    if p.is_file():
        print(p.relative_to(DIST))
print("\nOK: dist/ contains the runtime files and nothing else")

In [ ]:
import json, tarfile

ARCHIVE = AIP_ROOT / f"{AIP_ID}_dist.tgz"
ARCHIVE.unlink(missing_ok=True)
run(["eais-tools", "aip", "export", "--uuid", AIP_ID])
assert ARCHIVE.is_file() and ARCHIVE.stat().st_size, "export did not create a package"
with tarfile.open(ARCHIVE, "r:gz") as archive:
    members = {}
    for member in archive.getmembers():
        path = Path(member.name)
        assert not path.is_absolute() and ".." not in path.parts, member.name
        assert member.isfile() or member.isdir(), f"unexpected archive entry: {member.name}"
        if member.isfile():
            name = path.relative_to("dist").as_posix()
            assert name not in members, f"duplicate archive entry: {member.name}"
            members[name] = member
    dist_files = {p.relative_to(DIST).as_posix(): p for p in DIST.rglob("*") if p.is_file()}
    assert set(members) == set(dist_files), "archive file set differs from validated dist/"
    for name, path in dist_files.items():
        with archive.extractfile(members[name]) as source:
            assert hashlib.sha256(source.read()).hexdigest() == sha256(path), name
    with archive.extractfile(members["manifest.json"]) as source:
        packaged_manifest = json.load(source)
    assert packaged_manifest == json.loads((AIP_ROOT / "manifest.json").read_text())
print(f"OK: verified package {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.1f} MB)")
print(f"SHA256: {sha256(ARCHIVE)}")

## 4. Register the AI processor

The AIP folder is already in the station's `/data/aip`, so instead of uploading an archive we ask
the video-streaming service (ms3) to **scan** the folder and then **register** the AIP. The upload
API (`POST /aiprocessor/upload`) is for stations in normal mode and is rejected in development
mode.

In [ ]:
import requests

MS3 = "http://ms3_videostreaming:8000"

requests.post(f"{MS3}/aiprocessor/scan", timeout=60).raise_for_status()
response = requests.get(f"{MS3}/aiprocessors", timeout=30)
response.raise_for_status()
aip = next(a for a in response.json() if a["unique_id"] == AIP_ID)
assert aip["version"] == packaged_manifest["version"], "ms3 discovered a different AIP version"
print(f"Found {aip['unique_id']} v{aip['version']} (pk {aip['pk']})")

r = requests.post(f"{MS3}/aiprocessor/{aip['pk']}/register", timeout=300)
r.raise_for_status()
print(r.json())
status = requests.get(f"{MS3}/aiprocessor/{aip['pk']}/status", timeout=30)
status.raise_for_status()
print(status.json())
print("Registration was requested; step 5 verifies that inference actually starts.")

## 5. Test a pipeline and check the callback output

`eais-tools pipeline test` uploads a sample video, creates a pipeline named `eais_*` with this AIP,
starts it, and records the callback messages to a file. The tool listens until interrupted, so the
cell stops it with `SIGINT` after `LISTEN_SECONDS` (exit code 124 is expected and handled). `--retain_resources`
keeps the pipeline running so the dashboard in step 6 has live data. Step 7 removes it.

The next cell then checks the messages against the contract documented in
[`doc/USER_MANUAL.md`](doc/USER_MANUAL.md#callback-output-metadata-contract): batches of
`{since, until, data}` records, where `data` maps each source ID to per-class statistics.

In [ ]:
DUMP = AIP_ROOT / "tmp" / "callback_dump.json"
DUMP.parent.mkdir(exist_ok=True)
DUMP.unlink(missing_ok=True)
LISTEN_SECONDS = 90
run(["timeout", "--signal=INT", "--kill-after=30", str(LISTEN_SECONDS),
     "eais-tools", "pipeline", "test", "--aip", AIP_ID, "--inputs_num", "1",
     "--callback_output", "--retain_resources", "--dump_file", str(DUMP)],
    env={**os.environ, "COLUMNS": "160"}, expected=(0, 124))

In [ ]:
import json

CLASSES = ["car", "person", "truck"]

# The dump file is a sequence of JSON arrays (one per received message), not a single document.
text, pos, records = DUMP.read_text(), 0, []
decoder = json.JSONDecoder()
while pos < len(text):
    if text[pos].isspace():
        pos += 1
        continue
    batch, pos = decoder.raw_decode(text, pos)
    records.extend(batch)
assert records, "no callback messages were received"

peak = {c: 0 for c in CLASSES}
for record in records:
    assert {"since", "until", "data"} <= record.keys(), record.keys()
    for source in record["data"].values():
        assert set(source["obj_counter"]) == set(CLASSES), source["obj_counter"].keys()
        assert {"objInROIcnt", "objLCCurrCnt"} <= source["analytics_obj_cnt"].keys()
        for c in CLASSES:
            peak[c] = max(peak[c], source["obj_counter"][c]["max_val"])

print(f"{len(records)} merge windows received, peak counts per frame: {peak}")
assert any(peak.values()), "no objects detected in the sample video"
print("OK: callback output matches the documented contract")

In [ ]:
response = requests.get(f"{MS3}/pipelines", timeout=30)
response.raise_for_status()
pipelines = [p for p in response.json() if p["name"] == "eais_pipeline"]
assert pipelines, "test pipeline not found; did the previous step use --retain_resources?"
test_pipeline = max(pipelines, key=lambda p: p["creation_datetime"])
PIPELINE_ID = test_pipeline["pk"]
print(f"Test pipeline: {test_pipeline['name']} (pk {PIPELINE_ID})")

## 6. Install the Node-RED flow and dashboard

The repository ships a Node-RED project (`logic/`) and its UIBuilder dashboard (`dashboard/`).
This step installs them as the Node-RED project **`nodered-aipv1emi0001-dashboard`** in
`/data/projects`, connected to the test pipeline from step 5:

1. copy `logic/` to the project folder and `dashboard/` to its `uibuilder/` folder;
2. set the `AIP_PIPELINE_ID` environment variable of the **AIPV1EMI0001 Logic** tab;
3. install the dashboard's browser libraries (Vue, hls.js) with `npm ci`, so the page works
   without internet access afterwards (this step itself needs access to the npm registry);
4. make the folder a git repository, which Node-RED requires for projects.

The Node-RED nodes themselves (`@edge-ai/node-red-contrib-eais`, `node-red-contrib-uibuilder`)
are already installed on the station.

This step **does not switch** the station's active Node-RED project. Set `OVERWRITE = True` to
replace an earlier install; `flows_cred.json` is never copied.

In [ ]:
NR_PROJECT = "nodered-aipv1emi0001-dashboard"
OVERWRITE = False

project = Path("/data/projects") / NR_PROJECT
assert project.parent.is_dir(), "/data/projects is not mounted in this container"
if project.exists():
    assert OVERWRITE, f"{project} already exists; set OVERWRITE = True to replace it"
    shutil.rmtree(project)

project.mkdir()
for name in ("flows.json", "package.json", "package-lock.json", "README.md"):
    shutil.copy2(AIP_ROOT / "logic" / name, project / name)
shutil.copytree(AIP_ROOT / "dashboard", project / "uibuilder",
                ignore=shutil.ignore_patterns("node_modules", "README.md"))
(project / ".gitignore").write_text("node_modules/\nflows_cred.json\n.flows*.backup\n")

flows = json.loads((project / "flows.json").read_text())
logic_tab = next(n for n in flows if n["type"] == "tab" and n["label"] == "AIPV1EMI0001 Logic")
logic_tab["env"] = [{"name": "AIP_PIPELINE_ID", "value": PIPELINE_ID, "type": "str"}]
(project / "flows.json").write_text(json.dumps(flows, indent=4) + "\n")

subprocess.run(["npm", "ci", "--omit=dev", "--no-audit", "--no-fund"], cwd=project / "uibuilder", check=True)
git = ["git", "-c", "user.name=EdgeAI Station", "-c", "user.email=edgeai@localhost"]
subprocess.run(["git", "init", "-q", "-b", "main"], cwd=project, check=True)
subprocess.run(["git", "add", "-A"], cwd=project, check=True)
subprocess.run(git + ["commit", "-q", "-m", f"Install {AIP_ID} dashboard"], cwd=project, check=True)

for lib in ("vue", "hls.js"):
    assert (project / "uibuilder" / "node_modules" / lib / "package.json").is_file(), f"{lib} not installed"
print(f"OK: Node-RED project installed in {project}, wired to pipeline {PIPELINE_ID}")

### Open the project in Node-RED

These steps are manual on purpose, because opening a project replaces the flows currently running
in Node-RED for every user of the station.

1. Open the Node-RED editor from the station web UI.
2. Menu (☰) → **Projects** → **Open** → `nodered-aipv1emi0001-dashboard`.
3. Open the dashboard at `https://<station>/ms5/aipv1emi0001_dash/`. Counts for `car`, `person` and
   `truck` update every second, and the video plays from the pipeline's HLS stream.
4. To give other station users access, assign the `aipv1emi0001_dash` dashboard to them in
   **Accounts management**.

To follow another pipeline later, change `AIP_PIPELINE_ID` in the **AIPV1EMI0001 Logic** tab's
properties and deploy. If the callback node or ms3 is upgraded, enable the disabled
**inspect complete callback msg** debug node and compare a live message with the user manual.

## 7. Cleanup (keep the distributable package)

`pipeline clean` stops and deletes the `eais_*` test pipeline and its sample video, so the dashboard
stops receiving data until you point `AIP_PIPELINE_ID` at another pipeline.

The verified `AIPV1EMI0001_dist.tgz` archive and `dist/` are kept. Download the archive from
Jupyter to install it on another compatible station. Only the downloaded ONNX and build timing
cache are removed. Rebuilding later downloads the ONNX again.

Do **not** delete `models/.../1/yolov7-fp16-1x8x8.engine` or `models/.../libnvds_infercustomparser.so`
while the AIP is registered: ms3 copies the AIP folder each time a pipeline starts, so later starts
would fail.

In [ ]:
run(["eais-tools", "pipeline", "clean"])
for path in (MODEL_DIR / "yolov7tiny-parking-rev3-640.onnx", AIP_ROOT / "bin" / "timing.cache"):
    path.unlink(missing_ok=True)
assert ARCHIVE.is_file(), "the distributable package must remain after cleanup"
print(f"Package preserved: {ARCHIVE}\nSHA256: {sha256(ARCHIVE)}")